In [ ]:
!pip install gensim

In [2]:
import re
import torch
from torch import nn, optim
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
import gensim.downloader as api

In [18]:
# Get the word embeddings from gensim
glove = api.load('glove-wiki-gigaword-50')

[==================================================] 100.0% 66.0/66.0MB downloaded


In [2]:
# Use GPU instead of CPU for faster results
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
dataset = load_dataset('imdb')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [5]:
# Data is in the shape - review : label
# 25k samples in training set 12.5k positive, 12.5k negative
# 0 - negative, 1 - positive
dataset['train'][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [5]:
# Function to clean up text and return it word by word (token)
def tokenize(text):
  text = re.sub(r'<.*?>', ' ', text)  # remove HTML tags
  text = text.lower()
  text = text.replace("'", " '").replace("!", " !").replace("?", " ?")
  return text.split()

In [8]:
tokenize(dataset['train']['text'][0])[:10]

['i',
 'rented',
 'i',
 'am',
 'curious-yellow',
 'from',
 'my',
 'video',
 'store',
 'because']

In [8]:
class Vocabulary():
  def __init__(self):
    self.wordmap = {'<pad>' : 0,
                    '<unk>' : 1}

  def build(self, sentences):
    for sentence in sentences:
      sentence = tokenize(sentence)
      for word in sentence:
        if word not in self.wordmap.keys():
          self.wordmap[word] = max(self.wordmap.values()) + 1

    return None

  def to_indices(self, text):
    indices = []
    for word in tokenize(text):
      if word not in self.wordmap.keys():
        indices.append(1)
      else:
        indices.append(self.wordmap[word])

    return indices

In [9]:
# Get the 10k most frequent words for the vocbulary
def extract_common_words(limit=10000):
  reviews = dataset['train']['text']
  word_counts = dict()

  for review in reviews:
    review = tokenize(review)
    for word in review:
      if word not in word_counts.keys():
        word_counts[word] = 1
      else:
        word_counts[word] += 1

  sorted_words = sorted(word_counts, key=word_counts.get, reverse=True)
  most_common_words = sorted_words[:limit]

  return most_common_words

In [10]:
# Create our vocabulary with the 10k most common words in IMDB reviews
words = extract_common_words()
vocab = Vocabulary()
vocab.build(words)

In [11]:
class SentimentModel(nn.Module): # Inherit from nn.Module
    # Define the architecture of the model
    def __init__(self):
      super().__init__()
      # Create the embedding table (10000 rows, 1 for each word, each with 50 dimensions)
      self.embedding = nn.Embedding(
                          num_embeddings=len(vocab.wordmap),
                          embedding_dim=50
                      )
      self.linear = nn.Linear(50, 1)

    # Define the process of the model
    def forward(self, x):
        x = self.embedding(x)
        x = x.mean(dim=1) # get the average vector that consists of the means of d0, d1, ..., d49
        x = torch.sigmoid(self.linear(x)) # apply a sigmoid function, to get a value from 0 to 1 for classification

        return x

In [12]:
def get_review_indices(review):
  indices = vocab.to_indices(review)
  if len(indices) > 256:
    indices = indices[:256]
  else:
    indices += [0] * (256 - len(indices))  # 0 = <pad> index
  return indices

In [45]:
# Prepare data for training/testing
review_tensors_train = []
labels_train = dataset['train']['label']

review_tensors_test = []
labels_test = dataset['test']['label']

for review in dataset['train']['text']:
  review_tensors_train.append(torch.tensor(get_review_indices(review), dtype=torch.long))

for review in dataset['test']['text']:
  review_tensors_test.append(torch.tensor(get_review_indices(review), dtype=torch.long))

X_train = torch.stack(review_tensors_train)
y_train = torch.tensor(labels_train, dtype=torch.float32).reshape(-1, 1)

X_test = torch.stack(review_tensors_test)
y_test = torch.tensor(labels_test, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [49]:
# Create the model
model = SentimentModel()
model = model.to(device)

In [50]:
# Set the embedding tables to existing words in our table
# To gensim's values for a better optimized start
for word, idx in vocab.wordmap.items():
    if word in glove:
        model.embedding.weight.data[idx] = torch.tensor(glove[word])

In [51]:
# Define the loss function and the optimizer
loss_ft = nn.BCELoss()
opt = optim.Adam(model.parameters(), lr=0.001)

In [52]:
n_epochs = 40

for epoch in range(n_epochs):

  batch_loss = 0

  # Train
  model.train()
  for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    pred = model(X_batch)
    loss = loss_ft(pred, y_batch)
    loss.backward()
    opt.step()
    opt.zero_grad()

    batch_loss += loss.item()

  avg_batch_loss = batch_loss / len(train_loader)

  if epoch % (n_epochs / 10) == 0 or epoch == n_epochs - 1:
    print(f'Epoch {epoch}')
    print(f'Loss: {avg_batch_loss:.4f}')

Epoch 0
Loss: 0.6785
Epoch 4
Loss: 0.3960
Epoch 8
Loss: 0.2916
Epoch 12
Loss: 0.2421
Epoch 16
Loss: 0.2091
Epoch 20
Loss: 0.1844
Epoch 24
Loss: 0.1639
Epoch 28
Loss: 0.1469
Epoch 32
Loss: 0.1328
Epoch 36
Loss: 0.1201
Epoch 39
Loss: 0.1114


In [54]:
# Test

model.eval()
total_correct = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = (model(X_batch) > 0.5).float()
        total_correct += (preds == y_batch).sum().item()

print(f"Test accuracy: {total_correct / len(test_dataset):.4f}")

Test accuracy: 0.8589
